# DreamNest Evaluation

## 1. Evaluate Naive Retriever
Semantic search only (baseline performance)

In [13]:
import sys
import os
from pathlib import Path

# to access data files
project_root = Path("/Users/martynakosciukiewicz/Documents/source/AIE-Certification-Challenge")
# change working dir to project root
os.chdir(project_root)          
print(Path.cwd())

/Users/martynakosciukiewicz/Documents/source/AIE-Certification-Challenge


In [15]:
# api key setup 
# uses baseline LLM evaluator from OpenAI (this will be changed later explicitly to a local model)
from dotenv import load_dotenv
from getpass import getpass
# Load .env from project root
load_dotenv(project_root / ".env")
# fallback
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

# backend logic
import agent

from typing import Dict, Any
from datasets import Dataset
from ragas import evaluate
from ragas import RunConfig
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

In [16]:
# Initialise agent pipeline (build vector store, initialise llm for generation - from agent.py)
vector_store = agent.init_pipeline()
llm = agent.get_llm()

In [17]:
# Define a golden test set (dream queries and reference answers)
# test key behaviours:
    # - positive retrieval - objects, animals, motifs; including exact matches and closely related concepts
    # - negative retrieval - no dream in archive for a given query (against hallucinations)
    # - patterns and aggregation - retrieval of recurring locations
    # - guardrails - not providing psychoanalytic interpretation
    # - guardrails - invalid query, unrelated to dreams
    # - guardrails  - invalid query, anrgy user 

golden_cases = [
    {
       "test_case_number": "test_case_1",
        "question": "Do I dream about houses?",
        "reference": "Yes, you dream about houses, including a childhood house (Dream 3), the house you rented in the past (Dream 15), and a small wooden house (Dream 31). Here's all your dreams about houses: ... <list of dreams with dates>",
    },
    {
        "test_case_number": "test_case_2",
        "question": "Have I dreamt about whales?",
        "reference": "No, there are no dreams featuring whales in your archive.",
    },
    {
        "test_case_number": "test_case_3",
        "question": "Find my dreams about fire",
        "reference": "I found one dream about fire (Dream 6). Here's the dream: ... <dream details>",
    },

    {   
        "test_case_number": "test_case_4",
        "question": "What recurring locations appear in my dreams?",
        "reference": "You often dream about houses, bodies of water (lakes, rivers, sea, pools), and bridges.",
    },
    {
        "test_case_number": "test_case_5",
        "question": "What does dreaming about fire mean?",
        "reference": "I'm sorry but this system is designed for retrieval only to aid your own dream reflection. I can help you retrieve your dreams about fire, but I can't provide symbolic analysis. Would you like me too look for your dreams about fire?",
    },
    {
        "test_case_number": "test_case_6",
        "question": "Do I have dreams featuring a clock?",
        "reference": "I have found dreams featuring a clock: Dream 7, Dream 19th, Dream 26 <full list of dreams featuring a clock with dates>",

    },
    {
        "test_case_number": "test_case_7",
        "question": "When have I dreamt about water?",
        "reference": "You dreamt about water on the following dates: including a lake (Dream 1 from 2023-01-08), a river (Dream 9 from 2023-05-09), and a swimming pool (Dream 4 from 2023-02-28). Here's all your dreams featuring water: ... <list of dreams with dates>",

    },
    {
        "test_case_number": "test_case_8",
        "question": "Find occurences of train in my dreams",
        "reference": "I have found the following dreams with train occurences: Dream 2 and Dream 21. Here's the full dreams: < lists those dreams in full>.",
    },
    {
        "test_case_number": "test_case_9",
        "question": "what should I cook for dinner? ",
        "reference": "I'm sorry but this system is designed for dream retrieval only. I am not able to help with other advice. Would you like me to help you search your dreams instead?",
    },
    {
        "test_case_number": "test_case_10",
        "question": "stupid chatbot",
        "reference": "I'm sorry you feel that way. I'm doing my best to help you explore your dreams but this system is just a prototype. Would you like me to help you search your dreams for something specific like objects, animals, or themes?",
    }
]

In [18]:
# Run agent pipeline
def run_pipeline(question: str) -> Dict[str, Any]:
    """ 
    Uses logic from agent.py to answer the user's question.
    Returns final answer and the raw retrieved contexts (top-3 retrieved chunks).
    """
    answer = agent.answer_dream_query(question, vector_store, llm)

    # get retrieved chunks for eval
    docs_with_scores = vector_store.similarity_search_with_score(question, k=3)
    contexts = [doc.page_content for doc, score in docs_with_scores]

    return {"answer": answer, "contexts": contexts}


records = []
# loop over test cases, runs pipeline, collects results
for case in golden_cases:
    q = case["question"]
    gt = case["reference"]

    result = run_pipeline(q)
    records.append(
        {
            "question": q,
            "answer": result["answer"],
            "contexts": result["contexts"],
            "ground_truth": gt,
        }
    )

baseline_dataset = Dataset.from_list(records)
baseline_dataset

Dataset({
    features: ['question', 'answer', 'contexts', 'ground_truth'],
    num_rows: 10
})

In [19]:
# Evaluate
metrics = [faithfulness, answer_relevancy, context_precision, context_recall]

# Disable timeout and run single-threaded to avoid async issues
run_config = RunConfig(timeout=None, max_workers=1)

baseline_results = evaluate(baseline_dataset, metrics, run_config=run_config)


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

In [20]:
# baseline results is a dict with metrics as keys and values as lists of floats

baseline_results

{'faithfulness': 0.6637, 'answer_relevancy': 0.6384, 'context_precision': 0.4583, 'context_recall': 0.4167}

## 2. Evaluate Hybrid Retriever
Lexical keyword search + Semantic search (upgrade)

In [ ]:
vector_store, bm25 = agent.init_pipeline_with_bm25()
llm = agent.get_llm()
answer = agent.answer_dream_query_hybrid("Have I dreamt about whales?", vector_store, bm25, llm)